# Pancake Problem – Restricted 3-Generator Exploration (R_3 variant)

Instead of using all n−1 prefix reversals, we explore Cayley graphs with exactly **3 generators**:

| Generator | Description |
|-----------|-------------|
| **R_n** | Always included — reverses the full permutation |
| **R_2 or R_3** | Fixed small flip — choose which one (gives 2 graph families) |
| **R_k** | Varies from 2 (or 3) up to n — one line per k on each plot |

This mirrors the koltsov3 parameter sweep:
- **Dropdown** selects `(coset, R_2 vs R_3)` combination
- **Lines** show results for each value of k
- **x-axis** is n

## Cell 1: Imports and Setup

In [1]:
try:
    import numba
    print('Install CayleyPy without dependencies (for Kaggle CPU,GPU):'); print()
    !pip install git+https://github.com/cayleypy/cayleypy --no-deps -q
except:
    print('Install CayleyPy with dependencies (for Kaggle TPU):'); print()
    !pip install git+https://github.com/cayleypy/cayleypy -q

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import json
import os
from datetime import datetime
from tqdm.auto import tqdm
from cayleypy import CayleyGraph, CayleyGraphDef, PermutationGroups

print("Imports successful!")

Install CayleyPy without dependencies (for Kaggle CPU,GPU):



/home/ec2-user/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports successful!


## Cell 2: Configuration

In [11]:
MIN_N = 4       # need n >= 4 so R_n, R_2/R_3, R_k can all be distinct
MAX_N = 12      # for coset graphs; keep <= 12 for full graph
OUTPUT_DIR = "results_pancake3_r3"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FIXED_FLIPS = [2, 3]   # R_2 or R_3 as the fixed small flip

print(f"Config: n=[{MIN_N}, {MAX_N}], fixed flips={FIXED_FLIPS}")
print(f"Output directory: {OUTPUT_DIR}")

Config: n=[4, 12], fixed flips=[2, 3]
Output directory: results_pancake3_r3


## Cell 3: Generator Builder

For each `(n, fixed_flip, k)` we build exactly 3 generators:
- `R_n`: reverse all n positions
- `R_fixed_flip`: reverse first `fixed_flip` positions (2 or 3)
- `R_k`: reverse first `k` positions

If any two coincide (e.g. k == fixed_flip, or k == n), duplicates are dropped and the config is skipped.

In [3]:
def prefix_reversal(n, k):
    """Return the permutation that reverses the first k positions of [0..n-1]."""
    perm = list(range(n))
    perm[:k] = perm[:k][::-1]
    return perm

def make_group_def(n, k):
    """
    Build a CayleyGraphDef for generators {R_n, R_3, R_k}.
    Uses PermutationGroups.pancake(n) and replaces its generator list.
    R_3 is always included, R_n is always included, R_k varies (2 < k < n).
    Returns None if k<=3 or k>=n (degenerate).
    """
    if k <= 3 or k >= n:   # k must be strictly greater than 3 and less than n
        return None

    r_n = prefix_reversal(n, n)   # full reversal
    r_3 = prefix_reversal(n, 3)   # flip first three
    r_k = prefix_reversal(n, k)   # variable flip

    gens = [r_n, r_3, r_k]  # always 3 distinct since k!=3 and k!=n

    # Build via pancake template so we get the right internal structure,
    # then swap only the generators_permutations using dataclass replace
    import dataclasses
    template = PermutationGroups.pancake(n)
    return dataclasses.replace(template, generators_permutations=gens,
                               generator_names=[f'R{n}', 'R3', f'R{k}'],
                               name=f'pancake3r3_n{n}_k{k}')

# Sanity check
for n, k in [(6, 4), (5, 3), (4, 3), (4, 2), (5, 5)]:
    result = make_group_def(n, k)
    print(f"n={n}, k={k}: {'OK - ' + str(result.generators_permutations) if result else 'SKIPPED'}") 

n=6, k=4: OK - [[5, 4, 3, 2, 1, 0], [2, 1, 0, 3, 4, 5], [3, 2, 1, 0, 4, 5]]
n=5, k=3: SKIPPED
n=4, k=3: SKIPPED
n=4, k=2: SKIPPED
n=5, k=5: SKIPPED


## Cell 4: Coset Group Definitions

In [4]:
COSET_GROUPS = {
    "full_graph": {
        "FullGraph": lambda n: None,
    },
    "different": {
        "5Different": lambda n: list(range(4)) + [4]*(n-4) if n >= 5 else None,
        "6Different": lambda n: list(range(5)) + [5]*(n-5) if n >= 6 else None,
        "7Different": lambda n: list(range(6)) + [6]*(n-6) if n >= 7 else None,
        "8Different": lambda n: list(range(7)) + [7]*(n-7) if n >= 8 else None,
    },
    "then": {
        "Binary0then1":          lambda n: [0]*(n//2) + [1]*(n - n//2),
        "0then1then2":           lambda n: [0]*(n//3) + [1]*(n//3) + [2]*(n - 2*(n//3)),
        "0then1then2then3":      lambda n: [0]*(n//4) + [1]*(n//4) + [2]*(n//4) + [3]*(n - 3*(n//4)),
        "0then1then2then3then4": lambda n: [0]*(n//5) + [1]*(n//5) + [2]*(n//5) + [3]*(n//5) + [4]*(n - 4*(n//5)),
    },
    "coincide": {
        "2Coincide": lambda n: list(range(n-2)) + [n-2]*2 if n > 2 else None,
        "3Coincide": lambda n: list(range(n-3)) + [n-3]*3 if n > 3 else None,
        "4Coincide": lambda n: list(range(n-4)) + [n-4]*4 if n > 4 else None,
        "5Coincide": lambda n: list(range(n-5)) + [n-5]*5 if n > 5 else None,
        "6Coincide": lambda n: list(range(n-6)) + [n-6]*6 if n > 6 else None,
    },
    "repeats": {
        "Binary01Repeats":   lambda n: [0,1]*(n//2) + [0]*(n - 2*(n//2)),
        "Binary01Repeats_1": lambda n: [0,1]*(n//2) + [1]*(n - 2*(n//2)),
        "012Repeats":        lambda n: [0,1,2]*(n//3) + [0,1,2][:(n%3)],
        "011Repeats":        lambda n: [0,1,1]*(n//3) + [0,1,1][:(n%3)],
    },
}

print("Coset Groups Summary:")
for group_name, cosets in COSET_GROUPS.items():
    print(f"\n{group_name.upper()}:")
    for coset_name, func in cosets.items():
        example = func(9) if func(9) is not None else "None (full graph)"
        print(f"  {coset_name}: n=9 -> {example}")

Coset Groups Summary:

FULL_GRAPH:
  FullGraph: n=9 -> None (full graph)

DIFFERENT:
  5Different: n=9 -> [0, 1, 2, 3, 4, 4, 4, 4, 4]
  6Different: n=9 -> [0, 1, 2, 3, 4, 5, 5, 5, 5]
  7Different: n=9 -> [0, 1, 2, 3, 4, 5, 6, 6, 6]
  8Different: n=9 -> [0, 1, 2, 3, 4, 5, 6, 7, 7]

THEN:
  Binary0then1: n=9 -> [0, 0, 0, 0, 1, 1, 1, 1, 1]
  0then1then2: n=9 -> [0, 0, 0, 1, 1, 1, 2, 2, 2]
  0then1then2then3: n=9 -> [0, 0, 1, 1, 2, 2, 3, 3, 3]
  0then1then2then3then4: n=9 -> [0, 1, 2, 3, 4, 4, 4, 4, 4]

COINCIDE:
  2Coincide: n=9 -> [0, 1, 2, 3, 4, 5, 6, 7, 7]
  3Coincide: n=9 -> [0, 1, 2, 3, 4, 5, 6, 6, 6]
  4Coincide: n=9 -> [0, 1, 2, 3, 4, 5, 5, 5, 5]
  5Coincide: n=9 -> [0, 1, 2, 3, 4, 4, 4, 4, 4]
  6Coincide: n=9 -> [0, 1, 2, 3, 3, 3, 3, 3, 3]

REPEATS:
  Binary01Repeats: n=9 -> [0, 1, 0, 1, 0, 1, 0, 1, 0]
  Binary01Repeats_1: n=9 -> [0, 1, 0, 1, 0, 1, 0, 1, 1]
  012Repeats: n=9 -> [0, 1, 2, 0, 1, 2, 0, 1, 2]
  011Repeats: n=9 -> [0, 1, 1, 0, 1, 1, 0, 1, 1]


## Cell 5: Helper Functions

In [7]:
def is_valid_central(central):
    """Checks if central has >1 unique value."""
    return central is not None and len(np.unique(central)) > 1

def run_single_experiment(n, k, coset_name, coset_func):
    """Run BFS for generators {R_n, R_2, R_k}. Returns BfsResult or None."""
    try:
        defn = make_group_def(n, k)
        if defn is None:
            return None
        central = coset_func(n)
        if coset_name != 'FullGraph':
            if not is_valid_central(central):
                return None
            defn = defn.with_central_state(central)
        graph = CayleyGraph(defn, device="cpu")
        return graph.bfs(return_all_edges=False, return_all_hashes=False)
    except Exception as e:
        import traceback
        print(f"Error: {coset_name}, k=R_{k}, n={n}: {e}")
        traceback.print_exc()
        return None

def get_group_dir(group_name):
    group_dir = f"{OUTPUT_DIR}/{group_name}"
    os.makedirs(group_dir, exist_ok=True)
    return group_dir

def get_computed_combinations(group_name):
    """Return set of (coset, k, n) tuples already computed."""
    group_dir = get_group_dir(group_name)
    csv_path = f"{group_dir}/data.csv"
    if not os.path.exists(csv_path):
        return set()
    df = pd.read_csv(csv_path)
    return set(zip(df['coset'], df['k'], df['n']))

def run_group(group_name, cosets, min_n=MIN_N, max_n=MAX_N,
              k_range=None, coset_filter=None, skip_computed=True):
    """
    Run experiments with generators {R_n, R_2, R_k}, sweeping k and n.
    k ranges from 3 to n-1 (strictly between 2 and n).
    """
    if k_range is None:
        k_values = list(range(4, max_n))  # 3 to max_n-1
    else:
        k_values = list(range(k_range[0], k_range[1] + 1))

    computed = get_computed_combinations(group_name) if skip_computed else set()

    if coset_filter is None:
        filtered_cosets = cosets
    elif isinstance(coset_filter, str):
        if coset_filter not in cosets:
            raise ValueError(f"Coset '{coset_filter}' not found. Available: {list(cosets.keys())}")
        filtered_cosets = {coset_filter: cosets[coset_filter]}
    else:
        filtered_cosets = {c: v for c, v in cosets.items() if c in coset_filter}

    results = {
        'metadata': {
            'graph': 'pancake3',
            'group': group_name,
            'timestamp': datetime.now().isoformat(),
            'n_range': [min_n, max_n],
            'k_range': k_range,
            'generators': 'R_n + R_3 + R_k',
        },
        'results': {}
    }

    total_new = 0

    for coset_name, coset_func in filtered_cosets.items():
        skipped = 0
        computed_count = 0
        results['results'][coset_name] = {}

        total_iters = len(k_values) * (max_n - min_n + 1)
        pbar = tqdm(total=total_iters, desc=f"{coset_name}", leave=True)

        for k in k_values:
            k_key = f"k={k}"
            results['results'][coset_name][k_key] = {}

            for n in range(min_n, max_n + 1):
                pbar.set_postfix({'k': k, 'n': n})
                pbar.update(1)

                if make_group_def(n, k) is None:
                    continue
                if (coset_name, k, n) in computed:
                    skipped += 1
                    continue

                result = run_single_experiment(n, k, coset_name, coset_func)
                if result is not None:
                    results['results'][coset_name][k_key][f'n={n}'] = {
                        'diameter': result.diameter(),
                        'growth': result.layer_sizes,
                        'last_layer_size': len(result.last_layer())
                    }
                    computed_count += 1

        pbar.close()
        total_new += computed_count
        print(f"  {coset_name}: Skipped {skipped} cached, computed {computed_count} new")

    print(f"Completed {group_name} ({total_new} new results)")
    return results

def save_results(group_name, results, cosets, append=True):
    """Save results to CSV."""
    group_dir = get_group_dir(group_name)
    csv_path = f"{group_dir}/data.csv"
    rows = []
    for coset, k_data in results['results'].items():
        for k_key, n_data in k_data.items():
            k_val = int(k_key.split('=')[1])
            for n_key, metrics in n_data.items():
                n_val = int(n_key.split('=')[1])
                rows.append({
                    'coset': coset,
                    'k': k_val,
                    'n': n_val,
                    'diameter': metrics['diameter'],
                    'last_layer_size': metrics['last_layer_size'],
                    'total_states': sum(metrics['growth']),
                    'growth': json.dumps(metrics['growth'])
                })
    df_new = pd.DataFrame(rows)
    if append and os.path.exists(csv_path):
        df_existing = pd.read_csv(csv_path)
        df = pd.concat([df_existing, df_new], ignore_index=True)
        df = df.drop_duplicates(subset=['coset', 'k', 'n'], keep='last')
    else:
        df = df_new
    def compute_central(row):
        coset_func = cosets.get(row['coset'])
        if coset_func is None:
            return None
        central = coset_func(int(row['n']))
        return json.dumps(central) if central is not None else None
    df['central'] = df.apply(compute_central, axis=1)
    df = df.sort_values(['coset', 'k', 'n']).reset_index(drop=True)
    df.to_csv(csv_path, index=False)
    print(f"Saved: {csv_path} ({len(df)} total rows, {len(df_new)} new)")
    return df

def load_results(group_name):
    """Load results from CSV."""
    group_dir = get_group_dir(group_name)
    df = pd.read_csv(f"{group_dir}/data.csv")
    df['growth'] = df['growth'].apply(json.loads)
    if 'central' in df.columns:
        df['central'] = df['central'].apply(lambda x: json.loads(x) if pd.notna(x) else None)
    return df

print('Helper functions defined!')


Helper functions defined!


## Cell 6: Plotting

Three plots matching koltsov3 structure:
- **Plot 1** – Diameter vs n: dropdown = `(coset, R_2 or R_3)`, one line per k
- **Plot 2** – Growth curves: dropdown = `(coset, fixed_flip, k)`, one line per n  
- **Plot 3** – Last layer size vs n: dropdown = `(coset, fixed_flip)`, one line per k

In [8]:
def plot_group_results(group_name, df):
    """Create interactive Plotly plots.
    Generators are always R_n + R_3 + R_k.
    Dropdown: coset | Lines: k values | x-axis: n
    """
    group_dir = get_group_dir(group_name)
    coset_names = sorted(df['coset'].unique())
    k_values    = sorted(df['k'].unique())

    df = df.copy()
    df['growth_parsed'] = df['growth'].apply(lambda x: json.loads(x) if isinstance(x, str) else x)

    # ===== PLOT 1: Diameter vs n =====
    # Dropdown: coset  |  Lines: k
    fig1 = go.Figure()
    trace_idx = 0
    trace_map1 = {}
    for coset_name in coset_names:
        trace_map1[coset_name] = []
        sub = df[df['coset'] == coset_name]
        for k_val in k_values:
            k_df = sub[sub['k'] == k_val].sort_values('n')
            if len(k_df) == 0:
                continue
            fig1.add_trace(go.Scatter(
                x=k_df['n'], y=k_df['diameter'],
                mode='lines+markers', name=f'R_{k_val}',
                visible=(coset_name == coset_names[0]),
                hovertemplate='n=%{x}<br>diameter=%{y}' + f'<br>R_n + R_2 + R_{k_val}'
            ))
            trace_map1[coset_name].append(trace_idx)
            trace_idx += 1
    total_traces1 = trace_idx
    buttons1 = []
    first_button = True
    for coset_name in coset_names:
        if not trace_map1.get(coset_name, []):
            continue
        visibility = [False] * total_traces1
        for idx in trace_map1[coset_name]:
            visibility[idx] = True
        sub = df[df['coset'] == coset_name]
        n_range = [sub['n'].min() - 0.5, sub['n'].max() + 0.5]
        d_range = [0, sub['diameter'].max() * 1.05]
        buttons1.append(dict(label=coset_name, method='update',
                             args=[{'visible': visibility},
                                   {'xaxis.range': n_range, 'yaxis.range': d_range}]))
        if first_button:
            init_n1, init_d1 = n_range, d_range
            first_button = False
    fig1.update_layout(
        title=dict(text=f'Diameter vs n – R_n + R_3 + R_k ({group_name})', y=0.95),
        xaxis_title='n', yaxis_title='Diameter',
        xaxis=dict(range=init_n1), yaxis=dict(range=init_d1),
        updatemenus=[dict(buttons=buttons1, direction='down',
                          x=0.0, xanchor='left', y=1.02, yanchor='bottom', showactive=True)],
        legend=dict(x=1.02, y=1), height=650, margin=dict(t=100)
    )
    fig1.write_html(f'{group_dir}/diameter.html')
    fig1.show()

    # ===== PLOT 2: Growth curves =====
    # Dropdown: (coset, k)  |  Lines: n
    fig2 = go.Figure()
    trace_map2 = {}
    trace_idx = 0
    for coset_name in coset_names:
        for k_val in k_values:
            key = (coset_name, k_val)
            trace_map2[key] = []
            sub = df[(df['coset'] == coset_name) & (df['k'] == k_val)].sort_values('n')
            for _, row in sub.iterrows():
                growth = row['growth_parsed']
                fig2.add_trace(go.Scatter(
                    x=list(range(len(growth))), y=growth,
                    mode='lines+markers', name=f"n={row['n']}",
                    visible=(coset_name == coset_names[0] and k_val == k_values[0]),
                    hovertemplate=f'distance=%{{x}}<br>layer=%{{y}}<br>n={row["n"]}'
                ))
                trace_map2[key].append(trace_idx)
                trace_idx += 1
    total_traces2 = trace_idx
    buttons2 = []
    first_button = True
    for coset_name in coset_names:
        for k_val in k_values:
            key = (coset_name, k_val)
            if not trace_map2.get(key, []):
                continue
            visibility = [False] * total_traces2
            for idx in trace_map2[key]:
                visibility[idx] = True
            sub = df[(df['coset'] == coset_name) & (df['k'] == k_val)]
            growths = sub['growth_parsed'].tolist()
            if growths:
                max_dist  = max(len(g) for g in growths)
                max_layer = max(max(g) for g in growths)
                min_layer = max(1, min(min(g) for g in growths if min(g) > 0))
                dist_range  = [-0.5, max_dist + 0.5]
                layer_range = [np.log10(min_layer * 0.5), np.log10(max_layer * 2)]
            else:
                dist_range, layer_range = [0, 10], [0, 6]
            buttons2.append(dict(label=f'{coset_name}, R_{k_val}', method='update',
                                 args=[{'visible': visibility},
                                       {'xaxis.range': dist_range, 'yaxis.range': layer_range}]))
            if first_button:
                init_dist2, init_layer2 = dist_range, layer_range
                first_button = False
    fig2.update_layout(
        title=dict(text=f'Growth Curves – R_n + R_3 + R_k ({group_name})', y=0.95),
        xaxis_title='Distance', yaxis_title='Layer Size', yaxis_type='log',
        xaxis=dict(range=init_dist2), yaxis=dict(range=init_layer2),
        updatemenus=[dict(buttons=buttons2, direction='down',
                          x=0.0, xanchor='left', y=1.02, yanchor='bottom', showactive=True)],
        legend=dict(x=1.02, y=1), height=650, margin=dict(t=100)
    )
    fig2.write_html(f'{group_dir}/growth.html')
    fig2.show()

    # ===== PLOT 3: Last layer size vs n =====
    # Dropdown: coset  |  Lines: k
    fig3 = go.Figure()
    trace_idx = 0
    trace_map3 = {}
    for coset_name in coset_names:
        trace_map3[coset_name] = []
        sub = df[df['coset'] == coset_name]
        for k_val in k_values:
            k_df = sub[sub['k'] == k_val].sort_values('n')
            if len(k_df) == 0:
                continue
            fig3.add_trace(go.Scatter(
                x=k_df['n'], y=k_df['last_layer_size'],
                mode='lines+markers', name=f'R_{k_val}',
                visible=(coset_name == coset_names[0]),
                hovertemplate='n=%{x}<br>last_layer=%{y}' + f'<br>R_n + R_2 + R_{k_val}'
            ))
            trace_map3[coset_name].append(trace_idx)
            trace_idx += 1
    total_traces3 = trace_idx
    buttons3 = []
    first_button = True
    for coset_name in coset_names:
        if not trace_map3.get(coset_name, []):
            continue
        visibility = [False] * total_traces3
        for idx in trace_map3[coset_name]:
            visibility[idx] = True
        sub = df[df['coset'] == coset_name]
        n_range = [sub['n'].min() - 0.5, sub['n'].max() + 0.5]
        ll_min  = max(1, sub['last_layer_size'].min())
        ll_max  = sub['last_layer_size'].max()
        ll_range = [np.log10(ll_min * 0.5), np.log10(ll_max * 2)]
        buttons3.append(dict(label=coset_name, method='update',
                             args=[{'visible': visibility},
                                   {'xaxis.range': n_range, 'yaxis.range': ll_range}]))
        if first_button:
            init_n3, init_ll3 = n_range, ll_range
            first_button = False
    fig3.update_layout(
        title=dict(text=f'Last Layer Size vs n – R_n + R_3 + R_k ({group_name})', y=0.95),
        xaxis_title='n', yaxis_title='Last Layer Size', yaxis_type='log',
        xaxis=dict(range=init_n3), yaxis=dict(range=init_ll3),
        updatemenus=[dict(buttons=buttons3, direction='down',
                          x=0.0, xanchor='left', y=1.02, yanchor='bottom', showactive=True)],
        legend=dict(x=1.02, y=1), height=650, margin=dict(t=100)
    )
    fig3.write_html(f'{group_dir}/lastlayer.html')
    fig3.show()
    print(f"Interactive plots saved to {group_dir}/")

print('Plot function defined!')


Plot function defined!


---
## Group 1: full_graph
No coset restriction. Keep max_n small (≤12) due to n! state space.

In [14]:
%%time
group_name = "full_graph"
results = run_group(group_name, COSET_GROUPS[group_name],
                    min_n=4, max_n=10)
df = save_results(group_name, results, COSET_GROUPS[group_name], append=True)
plot_group_results(group_name, df)
display(df.head(20))

FullGraph: 100%|██████████| 42/42 [00:00<00:00, 1838.24it/s, k=9, n=10]

  FullGraph: Skipped 21 cached, computed 0 new
Completed full_graph (0 new results)
Saved: results_pancake3_r3/full_graph/data.csv (21 total rows, 0 new)
CPU times: user 50.9 ms, sys: 20 ms, total: 70.9 ms
Wall time: 70.3 ms


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

---
## Group 2: different

In [15]:
%%time
group_name = "different"
results = run_group(group_name, COSET_GROUPS[group_name],
                    min_n=5, max_n=MAX_N)
df = save_results(group_name, results, COSET_GROUPS[group_name], append=True)
plot_group_results(group_name, df)
display(df.head(20))

5Different: 100%|██████████| 64/64 [00:00<00:00, 68.90it/s, k=11, n=12]


  5Different: Skipped 0 cached, computed 36 new


6Different: 100%|██████████| 64/64 [00:01<00:00, 60.30it/s, k=11, n=12]


  6Different: Skipped 0 cached, computed 35 new


7Different: 100%|██████████| 64/64 [00:01<00:00, 45.30it/s, k=11, n=12]


  7Different: Skipped 0 cached, computed 33 new


8Different: 100%|██████████| 64/64 [00:02<00:00, 21.87it/s, k=11, n=12]


  8Different: Skipped 0 cached, computed 30 new
Completed different (134 new results)
Saved: results_pancake3_r3/different/data.csv (134 total rows, 134 new)
CPU times: user 10.4 s, sys: 60.6 ms, total: 10.5 s
Wall time: 6.41 s


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

---
## Group 3: then

In [ ]:
%%time
group_name = "then"
results = run_group(group_name, COSET_GROUPS[group_name],
                    min_n=4, max_n=MAX_N)
df = save_results(group_name, results, COSET_GROUPS[group_name], append=True)
plot_group_results(group_name, df)
display(df.head(20))

---
## Group 4: coincide

In [ ]:
%%time
group_name = "coincide"
results = run_group(group_name, COSET_GROUPS[group_name],
                    min_n=4, max_n=MAX_N)
df = save_results(group_name, results, COSET_GROUPS[group_name], append=True)
plot_group_results(group_name, df)
display(df.head(20))

---
## Group 5: repeats

In [ ]:
%%time
group_name = "repeats"
results = run_group(group_name, COSET_GROUPS[group_name],
                    min_n=4, max_n=MAX_N)
df = save_results(group_name, results, COSET_GROUPS[group_name], append=True)
plot_group_results(group_name, df)
display(df.head(20))

---
## Summary: List all output files

In [ ]:
import glob
print("Output files generated:")
for f in sorted(glob.glob(f"{OUTPUT_DIR}/**/*", recursive=True)):
    if os.path.isfile(f):
        print(f"  {f} ({os.path.getsize(f):,} bytes)")